# Financial Risk Engineering & Portfolio Management Recipe

This recipe combines 3 `algebrax` tools to analyze financial portfolio risk and algorithmic trading:

1. **Algorithmic Trade Execution State Machine** (`algebrax.automata.simulate_dfa`):
   Simulates Deterministic Finite Automata (DFA) state transitions over market signal streams.
2. **Spectral Asset Centrality** (`algebrax.matrix.academic.eigen_centrality`):
   Computes dominant eigenvector centrality on cross-asset correlation matrices to identify systemic risk hubs.
3. **Return Expectation & Variance Paths** (`algebrax.semiring.VarianceSemiring` & `algebrax.matrix.core.power`):
   Evaluates multi-step path return expectation $E[X]$ and portfolio variance $\text{Var}(X) = E[X^2] - (E[X])^2$.

In [ ]:
from algebrax.automata import simulate_dfa
from algebrax.matrix.academic import eigen_centrality
from algebrax.matrix.core import power
from algebrax.semiring import VarianceSemiring

## 1. Algorithmic Trade State Machine (simulate_dfa)

States: Cash (0), Invested (1), Risk_Hedge (2).

In [ ]:
trading_dfa = {
    0: {'buy_signal': 1, 'hold': 0, 'risk_alert': 2},
    1: {'sell_signal': 0, 'risk_alert': 2, 'hold': 1},
    2: {'clear_alert': 0, 'hold': 2},
}

signals = ['buy_signal', 'hold', 'risk_alert', 'hold', 'clear_alert', 'buy_signal']
final_state = simulate_dfa(0, signals, trading_dfa)
print(f'Final Execution State: State {final_state} (Invested)')

## 2. Systemic Risk Asset Centrality (eigen_centrality)

Computes dominant eigenvector centrality $v = \lambda_{\max} M v$ across cross-asset correlation matrices.

In [ ]:
asset_correlation = {
    0: {0: 1.0, 1: 0.2, 2: 0.6, 3: 0.8},
    1: {0: 0.2, 1: 1.0, 2: 0.3, 3: 0.1},
    2: {0: 0.6, 1: 0.3, 2: 1.0, 3: 0.5},
    3: {0: 0.8, 1: 0.1, 2: 0.5, 3: 1.0},
}

asset_names = {0: 'Tech ETF', 1: 'Bond Index', 2: 'Commodities', 3: 'Crypto Index'}
centrality = eigen_centrality(asset_correlation)

print('Asset Spectral Centrality Scores:')
for aid, score in sorted(centrality.items(), key=lambda x: x[1], reverse=True):
    print(f'  Asset {aid} [{asset_names[aid]}]: {score:.4f}')

## 3. Multi-Step Return Expectation & Variance (VarianceSemiring)

Calculates expected return $E[X] = \frac{r}{p}$ and variance $\text{Var}(X) = \frac{t}{p} - (E[X])^2$ over multi-step market transition paths.

In [ ]:
variance_semiring = VarianceSemiring()

market_graph = {
    0: {1: (0.6, 2.4, 2.4, 9.6), 2: (0.4, 4.8, 4.8, 57.6)},
    1: {3: (1.0, 5.0, 5.0, 25.0)},
    2: {3: (1.0, 15.0, 15.0, 225.0)},
    3: {},
}

m2 = power(market_graph, 2, semiring=variance_semiring)
path_stats = m2.get(0, {}).get(3, variance_semiring.zero)

p, r, _, t = path_stats
exp_return = r / p if p else 0.0
var_return = (t / p) - (exp_return**2) if p else 0.0

print(f'Expected Return E[X]: {exp_return:.2f}%')
print(f'Return Variance Var(X): {var_return:.2f} (%^2)')
print(f'Volatilty StdDev sigma: {var_return**0.5:.2f}%')